# Nowcoder Track 3 Data Pipeline V3
## 数据分析面经：完整 1–20 页 HAR → 可复现结构化数据

**用途：MGS 4701 Track 3 — What actually happens in a BA job interview**

本 Notebook 专门针对你这次新的搜索词：

```text
数据分析面经
```

以及对应的完整 1–20 页 HAR 文件：

```text
www.nowcoder.com(2).har
```

### 这份 Notebook 做什么？

它**不会重新调用牛客搜索 API**。  
它只读取你已经通过浏览器 Network 正常翻页并导出的 `.har` 文件。

完整流程：

**浏览器搜索 → 翻 1–20 页 → 导出 HAR → Python 读取 HAR → 提取搜索结果 JSON → 检查页码完整性 → 提取 records → 去重 → BA/DA 面经初筛 → 输出 CSV**

### 对你这次上传的 HAR，实际检查到：

- 搜索词：`数据分析面经`
- 页码：1–20 全部存在
- 每页：20 条
- raw search records：400
- 去重后 unique posts：380
- 初步 BA/DA/Analytics interview candidates：275

这些数值只是用于帮助你核对本次运行结果；代码不会为了匹配这些数字而补数据。

In [ ]:
from pathlib import Path
import base64
import html
import json
import re
from datetime import datetime, timezone

import pandas as pd

print("✅ Imports ready")

## 1. Configuration

为了避免把你之前“商业分析面经”的 HAR 混进来，这个版本默认只读取：

```text
www.nowcoder.com(2).har
```

如果你把 HAR 放进 `raw_har/` 文件夹也可以。

如果这个文件名不存在，程序只会在“当前目录 + raw_har/”中自动使用**唯一的一个 HAR**；如果发现多个 HAR，会停止并提醒你明确指定文件名。

In [ ]:
# -------------------------
# Input configuration
# -------------------------
CURRENT_DIR = Path(".")
RAW_HAR_DIR = Path("raw_har")

INPUT_HAR_FILES = [
    "www.nowcoder.com(2).har",
]

EXPECTED_QUERY = "数据分析面经"
EXPECTED_PAGES = set(range(1, 21))

# -------------------------
# Output configuration
# -------------------------
OUTPUT_DIR = Path("output_数据分析面经")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOWCODER_BASE = "https://www.nowcoder.com"

SEARCH_ENDPOINT_MARKER = (
    "gw-c.nowcoder.com/api/sparta/pc/search"
)

# Reference values from the HAR uploaded in this chat.
# These are used only for QA messages, never to create/impute rows.
REFERENCE_RAW_RECORDS = 400
REFERENCE_UNIQUE_POSTS = 380
REFERENCE_RELEVANT_CANDIDATES = 275

print("Working directory:", CURRENT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("Expected search query:", EXPECTED_QUERY)

## 2. Helper functions

In [ ]:
def decode_har_content(content_obj):
    """
    Decode a HAR response.content object.
    Supports normal text and base64 bodies.
    """
    if not isinstance(content_obj, dict):
        return ""

    text = content_obj.get("text", "")
    encoding = content_obj.get("encoding")

    if not text:
        return ""

    if encoding == "base64":
        try:
            raw = base64.b64decode(text)
            return raw.decode(
                "utf-8",
                errors="replace",
            )
        except Exception:
            return ""

    return text


def ms_to_iso(value):
    """
    Convert Unix timestamp in ms (or seconds) to UTC text.
    """
    if value in (None, ""):
        return ""

    try:
        value = float(value)

        if value > 10_000_000_000:
            value = value / 1000

        dt = datetime.fromtimestamp(
            value,
            tz=timezone.utc,
        )

        return dt.strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )

    except Exception:
        return ""


def clean_text(value):
    """
    Convert simple HTML-ish content to plain text.
    """
    if value is None:
        return ""

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def choose_content_object(data):
    """
    Select the useful nested content object.

    Observed Nowcoder search-result shapes:
      - contentData -> POST
      - momentData  -> MOMENT

    Includes a generic fallback.
    """
    if not isinstance(data, dict):
        return {}, "OTHER", ""

    known_shapes = [
        ("contentData", "POST"),
        ("momentData", "MOMENT"),
    ]

    for key, label in known_shapes:
        obj = data.get(key)

        if isinstance(obj, dict) and obj:
            return obj, label, key

    # Generic fallback for future response variants.
    for key, value in data.items():
        if not isinstance(value, dict):
            continue

        looks_like_content = any(
            field in value
            for field in (
                "title",
                "newTitle",
                "content",
                "newContent",
                "uuid",
            )
        )

        has_identity = any(
            field in value
            for field in (
                "id",
                "uuid",
            )
        )

        if looks_like_content and has_identity:
            return value, key.upper(), key

    return {}, "OTHER", ""


def build_source_url(uuid_value):
    """
    Construct the public Nowcoder detail URL when UUID exists.
    """
    uuid_value = str(
        uuid_value or ""
    ).strip()

    if not uuid_value:
        return ""

    return (
        f"{NOWCODER_BASE}/feed/main/detail/"
        f"{uuid_value}"
    )

## 3. Locate the exact HAR file

这个版本优先寻找指定的：

```text
www.nowcoder.com(2).har
```

这样不会误把旧 HAR 一起解析。

In [ ]:
def candidate_paths_for_name(filename):
    return [
        CURRENT_DIR / filename,
        RAW_HAR_DIR / filename,
    ]


def find_input_har_files():
    selected = []

    # 1) Exact configured files first.
    for filename in INPUT_HAR_FILES:
        matches = [
            path
            for path in candidate_paths_for_name(filename)
            if path.exists()
        ]

        if matches:
            selected.append(matches[0])

    if selected:
        return selected

    # 2) Safe fallback:
    # use automatic discovery only if exactly one HAR exists.
    candidates = list(
        CURRENT_DIR.glob("*.har")
    )

    if RAW_HAR_DIR.exists():
        candidates.extend(
            RAW_HAR_DIR.glob("*.har")
        )

    # Deduplicate absolute paths.
    unique = []
    seen = set()

    for path in candidates:
        resolved = str(path.resolve())

        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)

    if len(unique) == 1:
        print(
            "⚠️ Exact configured HAR name not found. "
            "Using the only HAR file available:"
        )
        return unique

    if len(unique) == 0:
        raise FileNotFoundError(
            "No HAR file found. Put "
            "'www.nowcoder.com(2).har' next to this notebook "
            "or inside raw_har/."
        )

    raise RuntimeError(
        "Multiple HAR files were found, but the configured "
        "www.nowcoder.com(2).har was not found. "
        "Please rename the correct HAR or edit INPUT_HAR_FILES."
    )


har_files = find_input_har_files()

print("HAR files selected:", len(har_files))

for path in har_files:
    print(" -", path.resolve())

## 4. Parse one HAR file

程序只解析 HAR 中已经保存的这类搜索结果请求：

```text
POST .../api/sparta/pc/search
```

并读取其中的：

```text
data.records
```

In [ ]:
def parse_one_har(path):
    with path.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:
        har = json.load(f)

    extracted_records = []
    request_audit = []

    entries = (
        har.get("log", {})
        .get("entries", [])
    )

    for entry_index, entry in enumerate(entries):
        request = entry.get(
            "request",
            {},
        ) or {}

        response = entry.get(
            "response",
            {},
        ) or {}

        method = request.get(
            "method",
            "",
        )

        request_url = request.get(
            "url",
            "",
        )

        # Only the actual search-result POST requests.
        if method != "POST":
            continue

        if SEARCH_ENDPOINT_MARKER not in request_url:
            continue

        # -------------------------
        # Parse request payload
        # -------------------------
        post_data = request.get(
            "postData",
            {},
        ) or {}

        request_text = post_data.get(
            "text",
            "",
        )

        try:
            request_json = (
                json.loads(request_text)
                if request_text
                else {}
            )
        except Exception:
            request_json = {}

        # -------------------------
        # Parse saved response body
        # -------------------------
        response_content = (
            response.get(
                "content",
                {},
            )
            or {}
        )

        response_text = decode_har_content(
            response_content
        )

        if not response_text:
            continue

        try:
            response_json = json.loads(
                response_text
            )
        except Exception:
            continue

        data = (
            response_json.get("data")
            or {}
        )

        records = (
            data.get("records")
            or []
        )

        search_query = (
            request_json.get("query")
            or ""
        )

        requested_page = (
            request_json.get("page")
        )

        response_page = (
            data.get("current")
        )

        page_size = (
            data.get("size")
        )

        reported_total = (
            data.get("total")
        )

        reported_total_pages = (
            data.get("totalPage")
        )

        # -------------------------
        # Request-level audit row
        # -------------------------
        request_audit.append({
            "source_har":
                path.name,

            "har_entry_index":
                entry_index,

            "search_query":
                search_query,

            "requested_page":
                requested_page,

            "response_page":
                response_page,

            "page_size":
                page_size,

            "reported_total":
                reported_total,

            "reported_total_pages":
                reported_total_pages,

            "records_in_response":
                len(records),

            "short_page_flag":
                (
                    isinstance(
                        page_size,
                        (int, float),
                    )
                    and len(records) < page_size
                ),

            "http_status":
                response.get("status"),

            "request_url":
                request_url,
        })

        # -------------------------
        # Record-level rows
        # -------------------------
        for rank, record in enumerate(
            records,
            start=1,
        ):
            outer_data = (
                record.get("data")
                or {}
            )

            user_brief = (
                outer_data.get("userBrief")
                or {}
            )

            (
                content_obj,
                record_kind,
                source_object,
            ) = choose_content_object(
                outer_data
            )

            content_id = str(
                content_obj.get("id")
                or outer_data.get("contentId")
                or ""
            )

            uuid_value = str(
                content_obj.get("uuid")
                or ""
            )

            title = clean_text(
                content_obj.get("title")
                or content_obj.get("newTitle")
                or record.get("title")
                or ""
            )

            content_text = clean_text(
                content_obj.get("content")
                or content_obj.get("newContent")
                or ""
            )

            author_id = str(
                content_obj.get("authorId")
                or content_obj.get("userId")
                or user_brief.get("userId")
                or ""
            )

            author_nickname = clean_text(
                user_brief.get("nickname")
                or ""
            )

            extracted_records.append({
                "source_har":
                    path.name,

                "search_query":
                    search_query,

                "page":
                    response_page,

                "rank_in_page":
                    rank,

                "record_kind":
                    record_kind,

                "source_object":
                    source_object,

                "rc_type":
                    record.get("rc_type"),

                "content_type":
                    outer_data.get("contentType"),

                "content_id":
                    content_id,

                "uuid":
                    uuid_value,

                "title":
                    title,

                "content":
                    content_text,

                "content_chars":
                    len(content_text),

                "author_id":
                    author_id,

                "author_nickname":
                    author_nickname,

                "create_time_utc":
                    ms_to_iso(
                        content_obj.get("createTime")
                    ),

                "show_time_utc":
                    ms_to_iso(
                        content_obj.get("showTime")
                    ),

                "source_url":
                    build_source_url(
                        uuid_value
                    ),

                "entity_data_id":
                    record.get("entityDataId"),
            })

    return extracted_records, request_audit

## 5. Parse the full 20-page HAR

In [ ]:
all_records = []
all_requests = []

for path in har_files:
    records, requests = parse_one_har(
        path
    )

    all_records.extend(records)
    all_requests.extend(requests)

    print(
        f"{path.name}: "
        f"{len(requests)} search requests, "
        f"{len(records)} raw records"
    )

raw_df = pd.DataFrame(
    all_records
)

request_df = pd.DataFrame(
    all_requests
)

print("\n" + "=" * 70)
print("RAW EXTRACTION SUMMARY")
print("=" * 70)

print(
    "Search-result requests:",
    len(request_df),
)

print(
    "Raw records extracted:",
    len(raw_df),
)

if not request_df.empty:
    print(
        "Search queries found:",
        request_df[
            "search_query"
        ]
        .dropna()
        .unique()
        .tolist()
    )

## 6. Validate query and page coverage

对于这次数据，我们希望看到：

```text
query = 数据分析面经
pages = 1–20
20 records per page
raw total = 400
```

如果有缺页或少于 20 条的页，程序会明确报告，而不会补造数据。

In [ ]:
def sorted_int_values(series):
    values = []

    for value in series.dropna():
        try:
            values.append(
                int(value)
            )
        except Exception:
            pass

    return sorted(values)


if request_df.empty:
    raise RuntimeError(
        "No Nowcoder search-result requests were found "
        "inside the HAR."
    )

queries_found = (
    request_df[
        "search_query"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if EXPECTED_QUERY not in queries_found:
    print(
        "⚠️ Query mismatch."
    )
    print(
        "Expected:",
        EXPECTED_QUERY,
    )
    print(
        "Found:",
        queries_found,
    )
else:
    print(
        "✅ Expected query found:",
        EXPECTED_QUERY,
    )


coverage_rows = []

for query, group in request_df.groupby(
    "search_query",
    dropna=False,
):
    captured_pages_all = sorted_int_values(
        group[
            "response_page"
        ]
    )

    captured_pages_unique = sorted(
        set(
            captured_pages_all
        )
    )

    captured_page_set = set(
        captured_pages_unique
    )

    missing_pages = sorted(
        EXPECTED_PAGES
        - captured_page_set
    )

    duplicate_pages = sorted(
        {
            page
            for page in captured_pages_all
            if captured_pages_all.count(page) > 1
        }
    )

    short_pages = sorted(
        set(
            sorted_int_values(
                group.loc[
                    group[
                        "short_page_flag"
                    ].fillna(False),
                    "response_page",
                ]
            )
        )
    )

    coverage_rows.append({
        "search_query":
            query,

        "unique_pages_captured":
            len(
                captured_pages_unique
            ),

        "captured_pages":
            ",".join(
                map(
                    str,
                    captured_pages_unique,
                )
            ),

        "missing_pages_1_to_20":
            ",".join(
                map(
                    str,
                    missing_pages,
                )
            ),

        "duplicate_page_captures":
            ",".join(
                map(
                    str,
                    duplicate_pages,
                )
            ),

        "short_pages":
            ",".join(
                map(
                    str,
                    short_pages,
                )
            ),

        "captured_records_from_requests":
            int(
                group[
                    "records_in_response"
                ].sum()
            ),

        "max_reported_total":
            group[
                "reported_total"
            ].max(),

        "max_reported_total_pages":
            group[
                "reported_total_pages"
            ].max(),

        "all_expected_pages_captured":
            len(
                missing_pages
            ) == 0,
    })


coverage_df = pd.DataFrame(
    coverage_rows
)

display(
    coverage_df
)

print("\nRequest-by-request audit:")

audit_columns = [
    "source_har",
    "search_query",
    "requested_page",
    "response_page",
    "page_size",
    "reported_total",
    "reported_total_pages",
    "records_in_response",
    "short_page_flag",
    "http_status",
]

display(
    request_df[
        audit_columns
    ]
    .sort_values(
        [
            "search_query",
            "response_page",
        ]
    )
)

## 7. Deduplicate posts

同一个帖子有时会在不同页/搜索结果形态中重复出现。

去重优先级：

1. UUID
2. content ID
3. fallback = title + author ID

In [ ]:
def make_dedupe_key(row):
    uuid_value = str(
        row.get(
            "uuid",
            "",
        )
        or ""
    ).strip()

    content_id = str(
        row.get(
            "content_id",
            "",
        )
        or ""
    ).strip()

    title = str(
        row.get(
            "title",
            "",
        )
        or ""
    ).strip()

    author_id = str(
        row.get(
            "author_id",
            "",
        )
        or ""
    ).strip()

    if uuid_value:
        return (
            "uuid:"
            + uuid_value
        )

    if content_id:
        return (
            "content_id:"
            + content_id
        )

    return (
        "fallback:"
        + title
        + "|"
        + author_id
    )


if not raw_df.empty:
    raw_df[
        "dedupe_key"
    ] = raw_df.apply(
        make_dedupe_key,
        axis=1,
    )

    hit_counts = (
        raw_df[
            "dedupe_key"
        ]
        .value_counts()
        .rename(
            "search_hit_count"
        )
    )

    unique_df = (
        raw_df
        .drop_duplicates(
            subset=[
                "dedupe_key"
            ],
            keep="first",
        )
        .copy()
    )

    unique_df[
        "search_hit_count"
    ] = unique_df[
        "dedupe_key"
    ].map(
        hit_counts
    )

else:
    unique_df = (
        raw_df.copy()
    )


print("=" * 70)
print("DEDUPLICATION")
print("=" * 70)

print(
    "Raw records:",
    len(raw_df),
)

print(
    "Unique posts:",
    len(unique_df),
)

print(
    "Duplicate search hits removed:",
    len(raw_df)
    - len(unique_df),
)

## 8. Preliminary Track 3 relevance screen

这一步只是**初步筛选候选样本**，不是最终人工 gold label。

Track 3 需要研究 BA、DA、Analytics 面试，所以这里要求：

- 至少命中一个 BA / DA / analytics 岗位相关词
- 至少命中一个面经 / 面试 / 校招 / 实习等面试相关词

In [ ]:
ROLE_TERMS = [
    # Data Analytics
    "数据分析",
    "数据分析师",
    "数分",
    "data analyst",
    "data analytics",

    # Business Analytics
    "商业分析",
    "商业分析师",
    "商分",
    "business analyst",
    "business analytics",

    # Strategy / Operations Analytics
    "策略分析",
    "策略分析师",
    "战略分析",
    "战略分析师",
    "经营分析",
    "经营分析师",
    "业务分析",
    "业务分析师",

    # Product / Growth Analytics
    "产品分析",
    "用户增长分析",
    "增长分析",
    "增长策略分析",

    # BI / generic analytics
    "商业智能",
    "business intelligence",
    "bi分析",
    "analytics",
]

INTERVIEW_TERMS = [
    "面经",
    "面试",
    "一面",
    "二面",
    "三面",
    "四面",
    "hr面",
    "hr 面",
    "群面",
    "电面",
    "电话面试",
    "笔试",
    "秋招",
    "春招",
    "校招",
    "暑期实习",
    "实习面试",
]


def find_hits(text, terms):
    text = str(
        text or ""
    ).lower()

    return [
        term
        for term in terms
        if term.lower() in text
    ]


def relevance_screen(row):
    combined_text = (
        str(
            row.get(
                "title",
                "",
            )
        )
        + "\n"
        + str(
            row.get(
                "content",
                "",
            )
        )
    )

    role_hits = find_hits(
        combined_text,
        ROLE_TERMS,
    )

    interview_hits = find_hits(
        combined_text,
        INTERVIEW_TERMS,
    )

    return pd.Series({
        "role_hits":
            "|".join(
                role_hits[:15]
            ),

        "interview_hits":
            "|".join(
                interview_hits[:15]
            ),

        "is_relevant_candidate":
            bool(
                role_hits
            )
            and bool(
                interview_hits
            ),
    })


if not unique_df.empty:
    relevance_columns = (
        unique_df.apply(
            relevance_screen,
            axis=1,
        )
    )

    unique_df = pd.concat(
        [
            unique_df.reset_index(
                drop=True
            ),
            relevance_columns.reset_index(
                drop=True
            ),
        ],
        axis=1,
    )

    relevant_df = unique_df[
        unique_df[
            "is_relevant_candidate"
        ]
        .fillna(False)
        .astype(bool)
    ].copy()

else:
    relevant_df = (
        unique_df.copy()
    )


print("=" * 70)
print("PRELIMINARY RELEVANCE SCREEN")
print("=" * 70)

print(
    "Unique posts:",
    len(unique_df),
)

print(
    "Track 3 relevant candidates:",
    len(relevant_df),
)

## 9. Quality checks

In [ ]:
def missing_count(df, column):
    if (
        df.empty
        or column not in df.columns
    ):
        return 0

    values = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    return int(
        values.eq("").sum()
    )


print("=" * 70)
print("QUALITY CHECKS")
print("=" * 70)

print(
    "Missing title:",
    missing_count(
        unique_df,
        "title",
    ),
)

print(
    "Missing content:",
    missing_count(
        unique_df,
        "content",
    ),
)

print(
    "Missing UUID:",
    missing_count(
        unique_df,
        "uuid",
    ),
)

print(
    "Missing source URL:",
    missing_count(
        unique_df,
        "source_url",
    ),
)


if not unique_df.empty:
    print(
        "\nRecord kind distribution:"
    )

    print(
        unique_df[
            "record_kind"
        ]
        .value_counts(
            dropna=False
        )
    )

## 10. Export this crawl's data

输出目录：

```text
output_数据分析面经/
```

会生成：

```text
nowcoder_数据分析面经_RAW.csv
nowcoder_数据分析面经_UNIQUE.csv
nowcoder_数据分析面经_RELEVANT_CANDIDATES.csv
nowcoder_数据分析面经_REQUEST_AUDIT.csv
nowcoder_数据分析面经_QUERY_COVERAGE.csv
```

In [ ]:
raw_output = (
    OUTPUT_DIR
    / "nowcoder_数据分析面经_RAW.csv"
)

unique_output = (
    OUTPUT_DIR
    / "nowcoder_数据分析面经_UNIQUE.csv"
)

relevant_output = (
    OUTPUT_DIR
    / "nowcoder_数据分析面经_RELEVANT_CANDIDATES.csv"
)

request_output = (
    OUTPUT_DIR
    / "nowcoder_数据分析面经_REQUEST_AUDIT.csv"
)

coverage_output = (
    OUTPUT_DIR
    / "nowcoder_数据分析面经_QUERY_COVERAGE.csv"
)


raw_df.to_csv(
    raw_output,
    index=False,
    encoding="utf-8-sig",
)

unique_df.to_csv(
    unique_output,
    index=False,
    encoding="utf-8-sig",
)

relevant_df.to_csv(
    relevant_output,
    index=False,
    encoding="utf-8-sig",
)

request_df.to_csv(
    request_output,
    index=False,
    encoding="utf-8-sig",
)

coverage_df.to_csv(
    coverage_output,
    index=False,
    encoding="utf-8-sig",
)


print("✅ Export complete")

for path in [
    raw_output,
    unique_output,
    relevant_output,
    request_output,
    coverage_output,
]:
    print(
        " -",
        path.resolve(),
    )

## 11. Final summary + reference QA

如果你运行的是本次上传的 `www.nowcoder.com(2).har`，正常应接近：

```text
Raw records = 400
Unique posts = 380
Relevant candidates = 275
```

如果不同，程序会提示，但不会强行修改数据。

In [ ]:
print("=" * 72)
print("数据分析面经 HAR PIPELINE — FINAL SUMMARY")
print("=" * 72)

print(
    "HAR files processed:",
    len(har_files),
)

print(
    "Search-result requests parsed:",
    len(request_df),
)

print(
    "Raw search records:",
    len(raw_df),
)

print(
    "Unique posts:",
    len(unique_df),
)

print(
    "Preliminary Track 3 relevant candidates:",
    len(relevant_df),
)

print()


def qa_compare(label, actual, reference):
    status = (
        "✅"
        if actual == reference
        else "⚠️"
    )

    print(
        f"{status} {label}: "
        f"actual={actual}, "
        f"reference={reference}"
    )


qa_compare(
    "Raw records",
    len(raw_df),
    REFERENCE_RAW_RECORDS,
)

qa_compare(
    "Unique posts",
    len(unique_df),
    REFERENCE_UNIQUE_POSTS,
)

qa_compare(
    "Relevant candidates",
    len(relevant_df),
    REFERENCE_RELEVANT_CANDIDATES,
)


print("\nPage coverage:")

for _, row in coverage_df.iterrows():
    print(
        "Query:",
        row[
            "search_query"
        ],
    )

    print(
        "  Pages captured:",
        row[
            "captured_pages"
        ],
    )

    print(
        "  Missing pages:",
        row[
            "missing_pages_1_to_20"
        ]
        or "None",
    )

    print(
        "  Short pages:",
        row[
            "short_pages"
        ]
        or "None",
    )

print(
    "\nReminder: "
    "is_relevant_candidate is only a preliminary "
    "rule-based screen and must be validated with the "
    "hand-coded gold set."
)

## 12. Preview this crawl's relevant records

In [ ]:
preview_columns = [
    "search_query",
    "page",
    "rank_in_page",
    "record_kind",
    "title",
    "content_chars",
    "role_hits",
    "interview_hits",
    "source_url",
]

preview_columns = [
    col
    for col in preview_columns
    if col in relevant_df.columns
]

if not relevant_df.empty:
    display(
        relevant_df[
            preview_columns
        ].head(30)
    )
else:
    print(
        "No relevant candidates found."
    )

# Methodology wording for class / README

> For the “数据分析面经” search term, we manually navigated pages 1–20 of the public Nowcoder search interface in a normal browser session and exported the browser Network log as a HAR file. Python then parsed the search-result JSON responses already stored in the HAR file, extracted record-level fields, audited page coverage, removed duplicate posts using UUID or content ID, and applied a transparent keyword rule only as a preliminary relevance screen. The original HAR file, notebook, raw dataset, cleaned unique dataset, and request audit are retained so that the processing workflow can be reproduced.

注意：自动 relevance filter 只是初筛，课程要求的最终 extraction / classification 仍需要用 hand-coded gold set 做 validation。